In [222]:
import pandas as pd

## Unemployment Data (Averaging the unemployment rates)

In [223]:
unemployment = pd.read_excel("Data/Unemployment_rate.xlsx", dtype={'State FIPS Code': str, 'County FIPS Code': str}, header=0)

In [224]:
unemployment.head()

,LAUS Code,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Labor Force,Employed,Unemployed,Unemploy-ment Rate (%)
0,CN0100100000000,01,001,"Autauga County, AL",Oct-24,28609,27798,811,2.8
1,CN0100300000000,01,003,"Baldwin County, AL",Oct-24,117918,114611,3307,2.8
2,CN0100500000000,01,005,"Barbour County, AL",Oct-24,8825,8451,374,4.2
3,CN0100700000000,01,007,"Bibb County, AL",Oct-24,8727,8447,280,3.2
4,CN0100900000000,01,009,"Blount County, AL",Oct-24,27134,26401,733,2.7


In [225]:
unemployment = unemployment.drop(["LAUS Code", "Labor Force", "Employed", "Unemployed"], axis=1)

In [226]:
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1
45094,NaN,NaN,NaN,NaN,"SOURCE: BLS, LAUS"
45095,NaN,NaN,NaN,NaN,"January 16, 2026"


In [227]:
unemployment = unemployment.drop([45094, 45095])
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45089,72,145,"Vega Baja Municipio, PR",Nov-25 p,5.5
45090,72,147,"Vieques Municipio, PR",Nov-25 p,4.9
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1


In [228]:
unemployment.dtypes

State FIPS Code                   object
County FIPS Code                  object
County Name/State Abbreviation    object
Period                            object
Unemploy-ment Rate (%)            object
dtype: object

In [229]:
# 1. Force everything to numeric. The '–' for all oct 25 entries will become NaN (null)
unemployment["Unemploy-ment Rate (%)"] = pd.to_numeric(unemployment["Unemploy-ment Rate (%)"], errors='coerce')

# 2. Drop the rows that are now NaN
unemployment = unemployment.dropna(subset=["Unemploy-ment Rate (%)"])

unemployment['full_fips'] = (
    unemployment['State FIPS Code'].astype(str).str.zfill(2) + 
    unemployment['County FIPS Code'].astype(str).str.zfill(3)
)

# 3. Now convert to int
unemployment["Unemploy-ment Rate (%)"] = unemployment["Unemploy-ment Rate (%)"].astype(int)

In [230]:
unemployment = unemployment.groupby(["full_fips", "County Name/State Abbreviation"]).agg(unemployment_rate = ("Unemploy-ment Rate (%)", "mean")).reset_index()

In [231]:
unemployment.tail()

,full_fips,County Name/State Abbreviation,unemployment_rate
3216,72145,"Vega Baja Municipio, PR",4.857143
3217,72147,"Vieques Municipio, PR",4.357143
3218,72149,"Villalba Municipio, PR",8.285714
3219,72151,"Yabucoa Municipio, PR",8.214286
3220,72153,"Yauco Municipio, PR",9.428571


In [232]:
state_map = {
    # 50 States
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    
    # District & Territories
    'DC': 'District of Columbia',
    'PR': 'Puerto Rico',
    'VI': 'Virgin Islands',
    'GU': 'Guam',
    'AS': 'American Samoa',
    'MP': 'Northern Mariana Islands'
}

In [233]:
# Split the column into two temporary parts
# n=1 ensures we only split at the last comma
split_data = unemployment['County Name/State Abbreviation'].str.split(',', n=1, expand=True)

# 2. Clean up the abbreviation (remove spaces)
# We use the 'state_map' dictionary we created earlier
state_abbr = split_data[1].str.strip().str.upper()
state_full = state_abbr.map(state_map)

# 3. Create the new combined column
# split_data[0] is the County Name
unemployment['County Name/State'] = split_data[0] + ", " + state_full

In [234]:
unemployment.head()

,full_fips,County Name/State Abbreviation,unemployment_rate,County Name/State
0,01001,"Autauga County, AL",2.153846,"Autauga County, Alabama"
1,01003,"Baldwin County, AL",2.307692,"Baldwin County, Alabama"
2,01005,"Barbour County, AL",3.538462,"Barbour County, Alabama"
3,01007,"Bibb County, AL",2.538462,"Bibb County, Alabama"
4,01009,"Blount County, AL",2.230769,"Blount County, Alabama"


In [235]:
unemployment["County Name/State"].isna().sum()

1

In [236]:
missingrows = unemployment[unemployment['County Name/State'].isna()]
print(missingrows)

    full_fips County Name/State Abbreviation  unemployment_rate  \
321     11001           District of Columbia           5.384615   

    County Name/State  
321               NaN  


In [237]:
# df.loc[condition, column] = value
unemployment.loc[unemployment['County Name/State Abbreviation'] == 'District of Columbia', 'County Name/State'] = 'District of Columbia, District of Columbia'

In [238]:
unemployment_final = unemployment.drop("County Name/State Abbreviation", axis=1)

unemployment_final['unemployment_rate'] = unemployment_final['unemployment_rate']/100

In [239]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State
3216,72145,0.048571,"Vega Baja Municipio, Puerto Rico"
3217,72147,0.043571,"Vieques Municipio, Puerto Rico"
3218,72149,0.082857,"Villalba Municipio, Puerto Rico"
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico"
3220,72153,0.094286,"Yauco Municipio, Puerto Rico"


In [240]:
unemployment_final["County Name/State"].isna().sum()

0

## Poverty Rate (dublicate rows and a few calculated columns)

In [241]:
poverty = pd.read_csv("Data/Poverty_rate.csv", header=0)

In [242]:
poverty.head()

,County,Population,Poverty,Poverty_Ugstudents,Unnamed: 4
0,"Autauga County, Alabama",NaN,NaN,NaN,NaN
1,Estimate,"57,409","6,473",134,NaN
2,"Baldwin County, Alabama",NaN,NaN,NaN,NaN
3,Estimate,"237,007","23,381",995,NaN
4,"Barbour County, Alabama",NaN,NaN,NaN,NaN


In [243]:
poverty = poverty.drop("Unnamed: 4", axis=1)

In [244]:
# bfill(axis=0) pulls data UP from the row below
poverty = poverty.bfill(axis=0, limit=1)

# Filter out the estimate rows
poverty = poverty[~poverty['County'].str.contains('estimate', case=False, na=False)]

In [245]:
poverty = poverty.reset_index()

In [246]:
poverty.tail()

,index,County,Population,Poverty,Poverty_Ugstudents
3217,6434,"Vega Baja Municipio, Puerto Rico","52,600","21,470",841
3218,6436,"Vieques Municipio, Puerto Rico","7,923","4,619",104
3219,6438,"Villalba Municipio, Puerto Rico","20,981","7,972",308
3220,6440,"Yabucoa Municipio, Puerto Rico","28,813","13,615",610
3221,6442,"Yauco Municipio, Puerto Rico","32,129","13,508",444


In [247]:
for i in ['Population', 'Poverty', 'Poverty_Ugstudents']:
    poverty[i] = pd.to_numeric(poverty[i].str.replace(',', ''), errors='coerce')

In [248]:
poverty['poverty_rate'] = (poverty['Poverty'] - poverty['Poverty_Ugstudents'])/poverty['Population']

In [249]:
poverty_final = poverty.drop(["Population", "Poverty", 'Poverty_Ugstudents'], axis = 1)

In [250]:
poverty_final.tail()

,index,County,poverty_rate
3217,6434,"Vega Baja Municipio, Puerto Rico",0.392186
3218,6436,"Vieques Municipio, Puerto Rico",0.569860
3219,6438,"Villalba Municipio, Puerto Rico",0.365283
3220,6440,"Yabucoa Municipio, Puerto Rico",0.451359
3221,6442,"Yauco Municipio, Puerto Rico",0.406611


In [251]:
poverty_final["County"].isna().sum()

0

### Checking for mismatches with county names in poverty and unemployment dfs as county names is the only common column to join them

In [252]:
missing_county =  set(unemployment_final['County Name/State']) - set(poverty['County'])
print(missing_county)

{'Guanica Municipio, Puerto Rico', 'Dona Ana County, New Mexico', 'Denver County/city, Colorado', 'Juana Diaz Municipio, Puerto Rico', 'Mayaguez Municipio, Puerto Rico', 'Anasco Municipio, Puerto Rico', 'Rio Grande Municipio, Puerto Rico', 'San Sebastian Municipio, Puerto Rico', 'Canovanas Municipio, Puerto Rico', 'Anchorage Borough/municipality, Alaska', 'Philadelphia County/city, Pennsylvania', 'Yakutat Borough/city, Alaska', 'Honolulu County/city, Hawaii', 'Las Marias Municipio, Puerto Rico', 'Sitka Borough/city, Alaska', 'Loiza Municipio, Puerto Rico', 'Nantucket County/town, Massachusetts', 'San Francisco County/city, California', 'Comerio Municipio, Puerto Rico', 'Catano Municipio, Puerto Rico', 'Juneau Borough/city, Alaska', 'Manati Municipio, Puerto Rico', 'Broomfield County/city, Colorado', 'Wrangell Borough/city, Alaska', 'San German Municipio, Puerto Rico', 'Bayamon Municipio, Puerto Rico', 'Penuelas Municipio, Puerto Rico', 'Rincon Municipio, Puerto Rico'}


In [253]:
missing_county =  set(poverty['County']) - set(unemployment_final['County Name/State'])
print(missing_county)

{'Añasco Municipio, Puerto Rico', 'Juneau City and Borough, Alaska', 'Rincón Municipio, Puerto Rico', 'Mayagüez Municipio, Puerto Rico', 'Canóvanas Municipio, Puerto Rico', 'San Francisco County, California', 'Kalawao County, Hawaii', 'Cataño Municipio, Puerto Rico', 'Loíza Municipio, Puerto Rico', 'Doña Ana County, New Mexico', 'Guánica Municipio, Puerto Rico', 'Wrangell City and Borough, Alaska', 'San Germán Municipio, Puerto Rico', 'Broomfield County, Colorado', 'Comerío Municipio, Puerto Rico', 'Yakutat City and Borough, Alaska', 'Philadelphia County, Pennsylvania', 'Bayamón Municipio, Puerto Rico', 'Peñuelas Municipio, Puerto Rico', 'Nantucket County, Massachusetts', 'Anchorage Municipality, Alaska', 'Juana Díaz Municipio, Puerto Rico', 'Río Grande Municipio, Puerto Rico', 'Honolulu County, Hawaii', 'Manatí Municipio, Puerto Rico', 'San Sebastián Municipio, Puerto Rico', 'Denver County, Colorado', 'Sitka City and Borough, Alaska', 'Las Marías Municipio, Puerto Rico'}


In [254]:
import unicodedata

def make_common_key(text):
    if pd.isna(text): return "nan"
    
    # 1. Standardize text
    text = str(text).lower().replace(',', '').replace('/', ' ')
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode("utf-8")
    
    # 2. PROTECT THE INDEPENDENT CITIES
    # These 6 cities have same-named counties. We detect "city" before it's stripped.
    cities_to_protect = ['baltimore', 'st. louis', 'fairfax', 'franklin', 'richmond', 'roanoke']
    is_independent_city = any(city in text for city in cities_to_protect) and "city" in text
    
    # 3. Strip "Noise" words
    noise_words = [
        "municipio", "municipality", "city and borough", "borough/city", 
        "county/city", "county/town", "borough", "town", "city", "county"
    ]
    for word in noise_words:
        text = text.replace(word, "")
    
    # Clean extra spaces
    cleaned_text = " ".join(text.split())
    
    # 4. APPEND SUFFIX if it was a protected city
    if is_independent_city:
        return f"{cleaned_text} city"
    
    return cleaned_text

In [255]:
# Apply to both DataFrames
unemployment_final['common_key'] = unemployment_final['County Name/State'].apply(make_common_key)
poverty_final['common_key'] = poverty_final['County'].apply(make_common_key)

In [256]:
missing_county =  set(unemployment_final['common_key']) - set(poverty_final['common_key'])
print(missing_county)

set()


In [257]:
missing_county =  set(poverty_final['common_key']) - set(unemployment_final['common_key']) 
print(missing_county)

{'kalawao hawaii'}


In [258]:
# Select rows where county is kalawao county, hawaii to check if it exists
filtered_df = poverty_final[poverty_final['County'] == 'Kalawao County, Hawaii']
filtered_df.head()

,index,County,poverty_rate,common_key
550,1100,"Kalawao County, Hawaii",0.227273,kalawao hawaii


In [259]:
# Select rows where county is maui county to check its unemployment rate as kalawao is mixed with that county
filtered_df2 = unemployment_final[unemployment_final['County Name/State'] == 'Maui County, Hawaii']
filtered_df2.head()

,full_fips,unemployment_rate,County Name/State,common_key
551,15009,0.025385,"Maui County, Hawaii",maui hawaii


In [260]:
# Add the respective row to unemployment_df, the fips code is 15005

# Define the values for Kalawao County
# Using Maui County's rate as the proxy per BLS/Census standard
new_row = {
    'full_fips': '15005',
    'unemployment_rate': 0.02538462,
    'County Name/State': 'Kalawao County, Hawaii',
    'common_key': 'kalawao hawaii'
}

# Insert into the DataFrame
# loc[len(df)] appends it to the very bottom
unemployment_final.loc[len(unemployment_final)] = new_row

In [261]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State,common_key
3217,72147,0.043571,"Vieques Municipio, Puerto Rico",vieques puerto rico
3218,72149,0.082857,"Villalba Municipio, Puerto Rico",villalba puerto rico
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico",yabucoa puerto rico
3220,72153,0.094286,"Yauco Municipio, Puerto Rico",yauco puerto rico
3221,15005,0.025385,"Kalawao County, Hawaii",kalawao hawaii


In [262]:
unemployment_final = unemployment_final.drop('County Name/State', axis=1)

In [263]:
poverty_final = poverty_final.drop(['County', 'index'], axis =1)

## Disability rate

In [264]:
disability = pd.read_csv("Data/Disability_rate.csv", header=0)

In [265]:
disability.head(10)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
1,Total,NaN
2,Estimate,"58,355"
3,With a disability,NaN
4,Estimate,"9,473"
5,Percent with a disability,NaN
6,Estimate,16.20%
7,"Baldwin County, Alabama",NaN
8,Total,NaN
9,Estimate,"243,859"


In [266]:
# bfill(axis=0) pulls data UP from the row below
disability = disability.bfill(axis=0, limit=1)

# Filter out the estimate rows
disability = disability[~disability['County'].str.contains('estimate', case=False, na=False)]
disability = disability[~disability['County'].str.contains('total', case=False, na=False)]
# Keep rows where the County is NOT exactly "with a disability", we did not use "str.contains" because in that case "percent with a disability" also gets deleted
disability = disability[disability['County'].str.strip() != 'With a disability']

In [267]:
disability.head(5)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
5,Percent with a disability,16.20%
7,"Baldwin County, Alabama",NaN
12,Percent with a disability,13.20%
14,"Barbour County, Alabama",NaN


In [268]:
# Getting the percentages aligned to the county names
disability = disability.bfill(axis=0, limit=1)

# Removing the redundant rows
disability = disability[disability['County'].str.strip() != 'Percent with a disability']

# renaming column name
disability.columns = ['County', 'disability_rate']

In [269]:
disability.dtypes

County             object
disability_rate    object
dtype: object

In [270]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",21.90%
22526,"Vieques Municipio, Puerto Rico",13.60%
22533,"Villalba Municipio, Puerto Rico",22.90%
22540,"Yabucoa Municipio, Puerto Rico",12.80%
22547,"Yauco Municipio, Puerto Rico",27.60%


In [271]:
disability['disability_rate'] = pd.to_numeric(disability['disability_rate'].astype(str).str.replace('%', ''), errors='coerce')
disability['disability_rate'] = disability['disability_rate']/100

In [272]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",0.219
22526,"Vieques Municipio, Puerto Rico",0.136
22533,"Villalba Municipio, Puerto Rico",0.229
22540,"Yabucoa Municipio, Puerto Rico",0.128
22547,"Yauco Municipio, Puerto Rico",0.276


In [273]:
disability["County"].isna().sum()

0

In [274]:
# Use the predefined function to make the county names same for all datasets
disability['common_key'] = disability['County'].apply(make_common_key)

# resetting index
disability_final = disability.reset_index()

In [275]:
disability_final.tail() 

,index,County,disability_rate,common_key
3217,22519,"Vega Baja Municipio, Puerto Rico",0.219,vega baja puerto rico
3218,22526,"Vieques Municipio, Puerto Rico",0.136,vieques puerto rico
3219,22533,"Villalba Municipio, Puerto Rico",0.229,villalba puerto rico
3220,22540,"Yabucoa Municipio, Puerto Rico",0.128,yabucoa puerto rico
3221,22547,"Yauco Municipio, Puerto Rico",0.276,yauco puerto rico


In [276]:
disability_final = disability_final.drop(['County', 'index'], axis = 1)

## Homeownership rate

In [277]:
homeownership = pd.read_csv("Data/Homeownership_rate.csv", header=0)

In [278]:
homeownership.tail()

,Label (Grouping),HOUSING TENURE!!Occupied housing units!!Owner-occupied
9661,Estimate,"8,597"
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9664,Estimate,"9,653"
9665,Percent,73.0%


In [279]:
homeownership.columns = ["County", "homeownership_rate"]

In [280]:
homeownership = homeownership[homeownership["County"].str.strip() != "Estimate"]

In [281]:
homeownership.tail()

,County,homeownership_rate
9659,Percent,77.9%
9660,"Yabucoa Municipio, Puerto Rico",NaN
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9665,Percent,73.0%


In [282]:
homeownership = homeownership.bfill(axis=0, limit=1)

homeownership = homeownership[homeownership["County"].str.strip() != "Percent"]

homeownership_final = homeownership.reset_index()

In [283]:
homeownership_final.tail()

,index,County,homeownership_rate
3217,9651,"Vega Baja Municipio, Puerto Rico",74.8%
3218,9654,"Vieques Municipio, Puerto Rico",66.8%
3219,9657,"Villalba Municipio, Puerto Rico",77.9%
3220,9660,"Yabucoa Municipio, Puerto Rico",70.7%
3221,9663,"Yauco Municipio, Puerto Rico",73.0%


In [284]:
homeownership_final["homeownership_rate"] = homeownership_final["homeownership_rate"].astype(str).str.replace("%", "", regex=False).astype(float)/100

In [285]:
homeownership_final['common_key'] = homeownership_final['County'].apply(make_common_key)

In [286]:
homeownership_final.head()

,index,County,homeownership_rate,common_key
0,0,"Autauga County, Alabama",0.771,autauga alabama
1,3,"Baldwin County, Alabama",0.776,baldwin alabama
2,6,"Barbour County, Alabama",0.682,barbour alabama
3,9,"Bibb County, Alabama",0.792,bibb alabama
4,12,"Blount County, Alabama",0.810,blount alabama


In [287]:
homeownership_final = homeownership_final.drop(['County', 'index'], axis=1)

## Average Meal Prices

In [288]:
amp = pd.read_csv("Data/average_meal_prices.csv", header=0)

In [289]:
amp.tail()

,"County, State",Cost Per Meal
3139,"Sweetwater County, Wyoming",$3.59
3140,"Teton County, Wyoming",$5.06
3141,"Uinta County, Wyoming",$3.45
3142,"Washakie County, Wyoming",$3.74
3143,"Weston County, Wyoming",$3.72


In [290]:
amp['Cost Per Meal ($)'] = amp["Cost Per Meal"].str.replace("$", "", regex=False).astype(float)

In [291]:
amp = amp.drop("Cost Per Meal", axis=1)

In [292]:
amp["common_key"] = amp["County, State"].apply(make_common_key)

In [293]:
amp.tail()

,"County, State",Cost Per Meal ($),common_key
3139,"Sweetwater County, Wyoming",3.59,sweetwater wyoming
3140,"Teton County, Wyoming",5.06,teton wyoming
3141,"Uinta County, Wyoming",3.45,uinta wyoming
3142,"Washakie County, Wyoming",3.74,washakie wyoming
3143,"Weston County, Wyoming",3.72,weston wyoming


In [294]:
missing_counties = set(poverty_final['common_key']) - set(amp['common_key'])
missing_counties

{'adjuntas puerto rico',
 'aguada puerto rico',
 'aguadilla puerto rico',
 'aguas buenas puerto rico',
 'aibonito puerto rico',
 'anasco puerto rico',
 'arecibo puerto rico',
 'arroyo puerto rico',
 'barceloneta puerto rico',
 'barranquitas puerto rico',
 'bayamon puerto rico',
 'cabo rojo puerto rico',
 'caguas puerto rico',
 'camuy puerto rico',
 'canovanas puerto rico',
 'carolina puerto rico',
 'catano puerto rico',
 'cayey puerto rico',
 'ceiba puerto rico',
 'ciales puerto rico',
 'cidra puerto rico',
 'coamo puerto rico',
 'comerio puerto rico',
 'corozal puerto rico',
 'culebra puerto rico',
 'dorado puerto rico',
 'fajardo puerto rico',
 'florida puerto rico',
 'guanica puerto rico',
 'guayama puerto rico',
 'guayanilla puerto rico',
 'guaynabo puerto rico',
 'gurabo puerto rico',
 'hatillo puerto rico',
 'hormigueros puerto rico',
 'humacao puerto rico',
 'isabela puerto rico',
 'jayuya puerto rico',
 'juana diaz puerto rico',
 'juncos puerto rico',
 'lajas puerto rico',
 'la

In [295]:
# Can ignore the below code as when we do inner join the puerto rico rows automatically will not be added to the final merged df

poverty_final = poverty_final[~poverty_final['common_key'].str.contains('puerto rico', case=False, na=False)]
homeownership_final = homeownership_final[~homeownership_final['common_key'].str.contains('puerto rico', case=False, na=False)]
unemployment_final = unemployment_final[~unemployment_final['common_key'].str.contains('puerto rico', case=False, na=False)]
disability_final = disability_final[~disability_final['common_key'].str.contains('puerto rico', case=False, na=False)]

In [296]:
amp_final = amp

In [297]:
print(len(poverty_final))
print(len(homeownership_final))
print(len(unemployment_final))
print(len(disability_final))
print(len(amp_final))

3144
3144
3144
3144
3144


## Population

In [298]:
pop = pd.read_csv("Data/population.csv", header=0)

In [299]:
pop.columns = ["County", "population"]

In [300]:
pop.head()

,County,population
0,"Autauga County, Alabama",NaN
1,Estimate,"59,947"
2,Margin of Error,*****
3,"Baldwin County, Alabama",NaN
4,Estimate,"246,989"


In [301]:
pop = pop[pop["County"].str.strip() != "Margin of Error"]

pop = pop.bfill(axis=0, limit=1)

pop = pop[pop["County"].str.strip() != "Estimate"]

pop = pop.reset_index()

In [302]:
pop['population'] = pop['population'].str.replace(',', '').astype(int)
pop.head()

,index,County,population
0,0,"Autauga County, Alabama",59947
1,3,"Baldwin County, Alabama",246989
2,6,"Barbour County, Alabama",24643
3,9,"Bibb County, Alabama",22130
4,12,"Blount County, Alabama",59518


In [303]:
pop['common_key'] = pop['County'].apply(make_common_key)
pop_final = pop.drop(["County","index"], axis=1)

pop_final.head()

,population,common_key
0,59947,autauga alabama
1,246989,baldwin alabama
2,24643,barbour alabama
3,22130,bibb alabama
4,59518,blount alabama


## Calculating Food Insecurity Rate

In [304]:
merged_df = ( unemployment_final.merge(poverty_final, on='common_key', how='inner')
                                .merge(homeownership_final, on='common_key', how='inner')
                                .merge(disability_final, on='common_key', how='inner')
                                .merge(amp_final, on='common_key', how='inner')
                                .merge(pop_final, on='common_key', how='inner')
            )

In [305]:
merged_df.head()

,full_fips,unemployment_rate,common_key,poverty_rate,homeownership_rate,disability_rate,"County, State",Cost Per Meal ($),population
0,01001,0.021538,autauga alabama,0.110418,0.771,0.162,"Autauga County, Alabama",3.64,59947
1,01003,0.023077,baldwin alabama,0.094453,0.776,0.132,"Baldwin County, Alabama",4.01,246989
2,01005,0.035385,barbour alabama,0.203246,0.682,0.194,"Barbour County, Alabama",3.45,24643
3,01007,0.025385,bibb alabama,0.214390,0.792,0.206,"Bibb County, Alabama",3.36,22130
4,01009,0.022308,blount alabama,0.124275,0.810,0.175,"Blount County, Alabama",3.36,59518


In [306]:
(merged_df == 0).sum()

full_fips             0
unemployment_rate     0
common_key            0
poverty_rate          0
homeownership_rate    1
disability_rate       0
County, State         0
Cost Per Meal ($)     0
population            0
dtype: int64

In [307]:
len(merged_df)

3144

In [308]:
# Split the 'County, State' column into two new columns
# expand=True turns the result into a DataFrame instead of a list
merged_df[['county', 'state']] = merged_df['County, State'].str.split(',', n=1, expand=True)

# 2. Clean up leading/trailing spaces (important for joining!)
merged_df['county'] = merged_df['county'].str.strip()
merged_df['state'] = merged_df['state'].str.strip()

In [309]:
merged_df = merged_df.drop("County, State", axis=1)

In [310]:
order = ["common_key", "full_fips", "county", "state", "population", "unemployment_rate", "poverty_rate", "disability_rate", "homeownership_rate", "Cost Per Meal ($)"]

merged_df = merged_df[order]

In [311]:
merged_df.head()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($)
0,autauga alabama,01001,Autauga County,Alabama,59947,0.021538,0.110418,0.162,0.771,3.64
1,baldwin alabama,01003,Baldwin County,Alabama,246989,0.023077,0.094453,0.132,0.776,4.01
2,barbour alabama,01005,Barbour County,Alabama,24643,0.035385,0.203246,0.194,0.682,3.45
3,bibb alabama,01007,Bibb County,Alabama,22130,0.025385,0.214390,0.206,0.792,3.36
4,blount alabama,01009,Blount County,Alabama,59518,0.022308,0.124275,0.175,0.810,3.36


In [312]:
# The final 2025/2023-based calculation
# Assuming rates are decimals (e.g., 0.12 for 12%)
merged_df['insecurity_rate'] = (
    0.101 + 0.013 + # Constant + 2023 Fixed Effect
    (merged_df['unemployment_rate'] * 0.460) +
    (merged_df['poverty_rate'] * 0.332) +
    (merged_df['disability_rate'] * 0.198) -
    (merged_df['homeownership_rate'] * 0.071)
)

In [313]:
merged_df.tail()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate
3139,teton wyoming,56039,Teton County,Wyoming,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460
3140,uinta wyoming,56041,Uinta County,Wyoming,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866
3141,washakie wyoming,56043,Washakie County,Wyoming,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282
3142,weston wyoming,56045,Weston County,Wyoming,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677
3143,kalawao hawaii,15005,Kalawao County,Hawaii,67,0.025385,0.227273,0.258,0.000,5.29,0.252215


In [314]:
distinct_count = merged_df['state'].nunique()

distinct_count

51

In [315]:
state_to_abbr = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'district of columbia': 'DC',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL',
    'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA',
    'maine': 'ME', 'maryland': 'MD', 'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN',
    'mississippi': 'MS', 'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV',
    'new hampshire': 'NH', 'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY',
    'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK', 'oregon': 'OR',
    'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC', 'south dakota': 'SD',
    'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT', 'virginia': 'VA',
    'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY'
}

In [316]:
# Ensure the column is lowercase and stripped to match the dictionary keys
merged_df['state_abbr'] = merged_df['state'].str.lower().str.strip().map(state_to_abbr)

# Check if any states failed to map (returns NaN if not in dictionary)
missing_states = merged_df[merged_df['state_abbr'].isna()]['state'].unique()
if len(missing_states) > 0:
    print(f"Warning: These values did not match the dictionary: {missing_states}")

In [317]:
merged_df['state'] = merged_df['state_abbr']

merged_df = merged_df.drop("state_abbr", axis=1)

In [318]:
region_map = {
    # Northeast (NE)
    'CT': 'NE', 'ME': 'NE', 'MA': 'NE', 'NH': 'NE', 'RI': 'NE', 'VT': 'NE',
    'NJ': 'NE', 'NY': 'NE', 'PA': 'NE',
    
    # Midwest (MW)
    'IL': 'MW', 'IN': 'MW', 'MI': 'MW', 'OH': 'MW', 'WI': 'MW',
    'IA': 'MW', 'KS': 'MW', 'MN': 'MW', 'MO': 'MW', 'NE': 'MW', 'ND': 'MW', 'SD': 'MW',
    
    # South (S)
    'DE': 'S', 'DC': 'S', 'FL': 'S', 'GA': 'S', 'MD': 'S', 'NC': 'S', 'SC': 'S', 'VA': 'S', 'WV': 'S',
    'AL': 'S', 'KY': 'S', 'MS': 'S', 'TN': 'S',
    'AR': 'S', 'LA': 'S', 'OK': 'S', 'TX': 'S',
    
    # West (W)
    'AZ': 'W', 'CO': 'W', 'ID': 'W', 'MT': 'W', 'NV': 'W', 'NM': 'W', 'UT': 'W', 'WY': 'W',
    'AK': 'W', 'CA': 'W', 'HI': 'W', 'OR': 'W', 'WA': 'W'
}


# Create the 'region' column
merged_df['region'] = merged_df['state'].map(region_map)

# Validation check: Ensure no state was missed
if merged_df['region'].isna().any():
    print("Warning: Some states were not assigned a region!")
    print(merged_df[merged_df['region'].isna()]['state'].unique())

In [319]:
merged_df.head()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S


In [326]:
import numpy as np

merged_df['food_insecure_population'] = np.ceil(merged_df['population']*merged_df['insecurity_rate']).astype(int)

In [327]:
merged_df

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region,food_insecure_population
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S,8267
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S,31371
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S,4627
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S,4015
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S,8491
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,teton wyoming,56039,Teton County,WY,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460,W,2632
3140,uinta wyoming,56041,Uinta County,WY,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866,W,2764
3141,washakie wyoming,56043,Washakie County,WY,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282,W,1019
3142,weston wyoming,56045,Weston County,WY,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677,W,824


## Annual Budget Shortfall

In [328]:
# Define Map the Meal Gap 2023-based constants
NATIONAL_AVG_MEAL_COST = 3.58       # National average meal cost
NATIONAL_AVG_WEEK_SHORTFALL = 22.37 # National average weekly shortfall
WEEKS_IN_YEAR = 52
MONTHS_OF_INSECURITY_FACTOR = 7/12  # Average duration factor per person

# Calculate localized weekly shortfall per person
# Logic: $22.37 * (Local Cost Per Meal / $3.58)
merged_df['weekly_shortfall_per_person'] = (
    NATIONAL_AVG_WEEK_SHORTFALL * (merged_df['Cost Per Meal ($)'] / NATIONAL_AVG_MEAL_COST)
)

# Calculate the total Annual Budget Shortfall
# Logic: Insecure Population * Local Weekly Shortfall * 52 weeks * 7/12 months
merged_df['annual_budget_shortfall'] = (
    merged_df['food_insecure_population'] * merged_df['weekly_shortfall_per_person'] * WEEKS_IN_YEAR * MONTHS_OF_INSECURITY_FACTOR
).round(0).astype(int)

# Drop the intermediate weekly column if not needed
final_df = merged_df.drop(columns=['weekly_shortfall_per_person'])

In [329]:
final_df

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region,food_insecure_population,annual_budget_shortfall
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S,8267,5703644
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S,31371,23843820
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S,4627,3025671
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S,4015,2556983
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S,8491,5407558
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,teton wyoming,56039,Teton County,WY,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460,W,2632,2524291
3140,uinta wyoming,56041,Uinta County,WY,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866,W,2764,1807425
3141,washakie wyoming,56043,Washakie County,WY,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282,W,1019,722352
3142,weston wyoming,56045,Weston County,WY,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677,W,824,580996
